# LeetCode 306 — Additive Number bằng Python

Link đề: https://leetcode.com/problems/additive-number/ · Độ khó: **Medium** · Chủ đề: String, Backtracking, Math (số lớn)

Skeleton LeetCode:

```python
class Solution:
    def isAdditiveNumber(self, num: str) -> bool:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium). Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (thử mọi cách cắt chuỗi thành từ `3` số trở lên rồi kiểm tra từng cách).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi chữ số là **số cộng tính (additive number)** nếu có thể **cắt thành ít nhất `3` số** (không có số `0` đứng đầu, trừ chính số `0`) sao cho **mỗi số từ số thứ ba trở đi bằng tổng hai số đứng ngay trước**.
2. Nhận ra **cấu trúc quyết định**: chỉ cần **chọn hai số đầu** (độ dài của chúng); toàn bộ phần còn lại bị **xác định duy nhất** (số kế tiếp luôn là tổng hai số trước), nên chỉ việc **kiểm tra**.
3. Đi từ **vét cạn mọi cách cắt** → **chọn hai số đầu rồi kiểm tra lặp** `O(n³)` → **quay lui (backtracking) đệ quy** → **cắt tỉa theo độ dài** → **cộng chuỗi (không đổi sang số nguyên)** để trả lời follow-up (tràn số).
4. Nắm các chi tiết dễ sai: **số `0` đứng đầu**, **tối thiểu `3` số**, số **rất lớn** (tới `35` chữ số — vượt `64-bit`; Python có số nguyên không giới hạn nhưng ta vẫn học cách cộng chuỗi).
5. Tránh các lỗi kinh điển: cho phép số có **`0` đứng đầu** (như `"03"`), chấp nhận dãy chỉ có **hai** số, so sánh **tiền tố** mà quên kiểm tra **hết chuỗi**, và giới hạn độ dài hai số đầu sai.

## 2. Đề bài & ví dụ

**Số cộng tính (additive number)** là chuỗi mà các chữ số của nó có thể tạo thành một **dãy cộng tính**.

Một dãy cộng tính hợp lệ phải có **ít nhất ba số**. Ngoại trừ hai số đầu tiên, mỗi số kế tiếp phải bằng **tổng của hai số đứng ngay trước nó**.

Cho một chuỗi chỉ chứa chữ số, trả về `true` nếu nó là số cộng tính, ngược lại `false`.

**Lưu ý:** các số trong dãy **không được có số `0` đứng đầu**, nên các dãy `1, 2, 03` hoặc `1, 02, 3` là **không hợp lệ**.

**Ví dụ 1**

- Input: `"112358"`
- Output: `true`
- Giải thích: các chữ số tạo thành dãy `1, 1, 2, 3, 5, 8`: `1+1=2`, `1+2=3`, `2+3=5`, `3+5=8`.

**Ví dụ 2**

- Input: `"199100199"`
- Output: `true`
- Giải thích: dãy `1, 99, 100, 199`: `1+99=100`, `99+100=199`.

**Ràng buộc:** `1 <= num.length <= 35`; `num` chỉ gồm chữ số.

**Follow up:** làm sao xử lý **tràn số** khi các số nguyên rất lớn?

In [1]:
from typing import List
from itertools import product
import random
import time

examples = [
    ("112358", True),
    ("199100199", True),
]
for num, expected in examples:
    print(f"num={num!r} -> kỳ vọng {expected}")

num='112358' -> kỳ vọng True
num='199100199' -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Cắt chuỗi (slicing) và đổi `str ↔ int`

`s[i:j]` lấy các ký tự từ `i` đến `j-1`; `int("099")` cho `99` (**mất số 0 đứng đầu**, vì vậy ta phải kiểm tra số 0 đứng đầu **trên chuỗi**, không trên số nguyên); `str(99)` cho `"99"`. `s.startswith(t, pos)` kiểm tra `s[pos:]` có bắt đầu bằng `t` không.
#### Code demo

In [2]:
s = "199100199"
print(s[0:1], s[1:3], s[3:6], s[6:])         # 1 99 100 199
print(int("099"), str(99), len(str(99)))      # 99 99 2
print(s.startswith("100", 3), s.startswith("101", 3))   # True False

1 99 100 199
99 99 2
True False


### 3.2. Số có số 0 đứng đầu

Số hợp lệ là `"0"` hoặc chuỗi **không bắt đầu bằng `'0'`**. Vậy `"0"`, `"10"`, `"907"` hợp lệ; `"00"`, `"03"`, `"0123"` không. Điều kiện: `len(t) == 1 or t[0] != "0"`.
#### Code demo

In [3]:
def valid_number(t):
    return len(t) == 1 or t[0] != "0"


for t in ["0", "10", "907", "00", "03", "0123", ""]:
    print(repr(t), valid_number(t) if t else "chuỗi rỗng: không hợp lệ")

'0' True
'10' True
'907' True
'00' False
'03' False
'0123' False
'' chuỗi rỗng: không hợp lệ


### 3.3. Dãy cộng tính được xác định bởi hai số đầu

Biết hai số đầu `a`, `b` thì số tiếp theo **buộc phải** là `a + b`, rồi `b + (a+b)`, ... — **không còn lựa chọn**. Như vậy ta chỉ cần **thử mọi cặp (độ dài số thứ nhất, độ dài số thứ hai)** rồi **kiểm tra** (dựng dãy và khớp với phần còn lại của chuỗi).
#### Code demo

In [4]:
a, b = 1, 99
seq = [a, b]
for _ in range(2):
    seq.append(seq[-1] + seq[-2])
print(seq, "->", "".join(map(str, seq)))       # [1, 99, 100, 199] -> 199100199

[1, 99, 100, 199] -> 199100199


### 3.4. Giới hạn độ dài hai số đầu

Gọi `n = len(num)`, độ dài số thứ nhất `i`, số thứ hai `j`, phần còn lại có độ dài `n - i - j` chứa **ít nhất một số** (số thứ ba) dài **không ngắn hơn** `max(i, j)` (tổng của hai số không nhỏ hơn mỗi số, mà không có số `0` đứng đầu nên số dài hơn hoặc bằng). Vậy `n - i - j >= max(i, j)`. Ngoài ra `i` chạy `1..n//2`.
#### Code demo

In [5]:
n = 9
valid_pairs = [(i, j) for i in range(1, n) for j in range(1, n) if n - i - j >= max(i, j)]
print(len(valid_pairs), "cặp độ dài hợp lệ so với", (n - 1) * (n - 1), "cặp tùy ý")

11 cặp độ dài hợp lệ so với 64 cặp tùy ý


### 3.5. Số lớn và cộng chuỗi

Chuỗi dài tới `35` chữ số cho các số tới `10^35`, vượt `int64`/`long` của C++/Java. Python tự xử lý số nguyên lớn; trong ngôn ngữ khác ta **cộng hai số dưới dạng chuỗi** từng chữ số (như cộng tay, nhớ `carry`). Ta sẽ cài đặt `add_strings` và dùng nó ở Cách 5.
#### Code demo

In [6]:
def add_strings(x: str, y: str) -> str:
    i, j, carry = len(x) - 1, len(y) - 1, 0
    digits = []
    while i >= 0 or j >= 0 or carry:
        total = carry
        if i >= 0:
            total += ord(x[i]) - 48
            i -= 1
        if j >= 0:
            total += ord(y[j]) - 48
            j -= 1
        digits.append(chr(total % 10 + 48))
        carry = total // 10
    return "".join(reversed(digits))


print(add_strings("99", "1"), add_strings("123456789012345678901234567890", "987654321098765432109876543210"))
assert add_strings("123456789012345678901234567890", "987654321098765432109876543210") == str(123456789012345678901234567890 + 987654321098765432109876543210)

100 1111111110111111111011111111100


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 2 `"199100199"` (`n = 9`). Thử các cặp độ dài số đầu:

- `(1, 1)`: `1` và `9` ⇒ tổng `10`; chuỗi còn lại `"9100199"` không bắt đầu bằng `"10"` ⇒ sai.
- `(1, 2)`: `1` và `99` ⇒ tổng `100`; phần còn lại `"100199"` bắt đầu bằng `"100"` ✓; tiếp theo `99 + 100 = 199`; phần còn lại `"199"` ✓ khớp hết ⇒ **hợp lệ**.

Ví dụ `"1023"`: `(1,1)`: `1+0=1`, phần còn lại `"23"` không bắt đầu bằng `"1"`; `(1,2)`: số thứ hai `"02"` có số 0 đứng đầu ⇒ loại; `(2,1)`: `10 + 2 = 12`, còn `"3"` ⇒ sai ⇒ `false`.

**Cách nghĩ:** chỉ có `O(n²)` cặp độ dài số đầu; mỗi cặp xác định duy nhất dãy mong đợi; ta so dãy đó với phần còn lại của chuỗi.

## 5. Cách 1 — Brute Force / baseline: thử mọi cách cắt chuỗi

**Ý tưởng.** Một cách cắt là chọn tập các vị trí cắt trong `n - 1` khe giữa các chữ số: `2^(n-1)` cách. Với mỗi cách cắt thành `k >= 3` số hợp lệ (không số 0 đứng đầu), kiểm tra từ số thứ ba trở đi bằng tổng hai số trước. Chỉ dùng được cho `n ≲ 16`.
#### Code demo

In [7]:
def additive_all_splits(num):
    n = len(num)
    for mask in range(1 << (n - 1)):                         # mỗi bit: có cắt ở khe giữa chữ số i và i + 1 không
        parts, start = [], 0
        for i in range(n - 1):
            if mask >> i & 1:
                parts.append(num[start:i + 1])
                start = i + 1
        parts.append(num[start:])
        if len(parts) < 3:
            continue
        if any(len(p) > 1 and p[0] == "0" for p in parts):
            continue
        values = [int(p) for p in parts]
        if all(values[k] == values[k - 1] + values[k - 2] for k in range(2, len(values))):
            return True
    return False


for num, expected in examples:
    print(num, "->", additive_all_splits(num), "(kỳ vọng", expected, ")")

112358 -> True (kỳ vọng True )
199100199 -> True (kỳ vọng True )


**Độ phức tạp (Cách 1).** **Time `O(2^n · n)`**; với `n = 35` là `~10^11` — bất khả thi; **Space `O(n)`**.

**Nút thắt.** Hầu hết các cách cắt vi phạm ngay ở số thứ ba. Vì hai số đầu quyết định toàn bộ phần còn lại, chỉ cần thử `O(n²)` cặp độ dài số đầu.

## 6. Ý tưởng thuật toán / bước đột phá

**Chọn hai số đầu rồi kiểm tra.** Với mỗi `i` (độ dài số thứ nhất) và `j` (độ dài số thứ hai) thỏa ràng buộc 3.4 và không có số 0 đứng đầu: `a = int(num[:i])`, `b = int(num[i:i+j])`, `pos = i + j`. Lặp: `c = a + b`, `s = str(c)`; nếu `num` tại `pos` **không** bắt đầu bằng `s` ⇒ thất bại; ngược lại `pos += len(s)`, `a, b = b, c`. Khi `pos == n` ⇒ thành công (đã đủ ba số vì ít nhất số thứ ba đã khớp).

Tổng cộng `O(n²)` cặp × `O(n)` kiểm tra mỗi cặp = `O(n³)`; với `n <= 35` là cỡ vài nghìn phép.

## 7. Các cách tiếp theo

### Cách 2 — Chọn hai số đầu và kiểm tra bằng vòng lặp (cách gọn nhất)

**Giải thích.** Như mục 6.
#### Code demo

In [8]:
def additive_two_first(num):
    n = len(num)
    for i in range(1, n):                                      # độ dài số thứ nhất
        if num[0] == "0" and i > 1:
            break                                              # số thứ nhất có số 0 đứng đầu: mọi i lớn hơn đều sai
        a = int(num[:i])
        for j in range(1, n - i):                              # độ dài số thứ hai
            if num[i] == "0" and j > 1:
                break                                          # số thứ hai có số 0 đứng đầu
            if n - i - j < max(i, j):
                continue                                       # phần còn lại không đủ chỗ cho số thứ ba
            b = int(num[i:i + j])
            pos, x, y = i + j, a, b
            while pos < n:
                z = x + y
                s = str(z)
                if not num.startswith(s, pos):
                    break
                pos += len(s)
                x, y = y, z
            if pos == n:
                return True
    return False


for num, expected in examples:
    print(num, "->", additive_two_first(num), "(kỳ vọng", expected, ")")

112358 -> True (kỳ vọng True )
199100199 -> True (kỳ vọng True )


**Độ phức tạp Cách 2:** `O(n²)` cặp `(i, j)`, mỗi cặp kiểm tra tối đa `O(n)` (mỗi bước tiêu thụ ít nhất một ký tự, `str(z)` và `startswith` tốn `O(độ dài số)`): **Time `O(n³)`** (worst), **Space `O(n)`**.

### Cách 3 — Quay lui đệ quy (cách nộp chuẩn)

**Giải thích.** `dfs(a, b, rest)`: nếu `rest` rỗng ⇒ thành công; ngược lại số kế tiếp phải bằng `a + b`; **thử mọi tiền tố** của `rest` làm số tiếp theo (không số 0 đứng đầu) và chỉ đi tiếp nếu nó bằng `a + b`. Hai số đầu được chọn bằng hai vòng ngoài.
#### Code demo

In [9]:
def additive_backtracking(num):
    n = len(num)

    def dfs(a, b, rest):
        if not rest:
            return True
        target = a + b
        if target > 0 and rest[0] == "0":                      # số tiếp theo (nếu > 0) không được có 0 đứng đầu
            return False
        for k in range(1, len(rest) + 1):
            value = int(rest[:k])
            if value == target:
                return dfs(b, value, rest[k:])
            if value > target:
                return False                                   # tiền tố đã lớn hơn mục tiêu: dài hơn chỉ lớn thêm
        return False

    for i in range(1, n - 1):
        if num[0] == "0" and i > 1:
            break
        for j in range(1, n - i):
            if num[i] == "0" and j > 1:
                break
            if dfs(int(num[:i]), int(num[i:i + j]), num[i + j:]):
                return True
    return False


for num, expected in examples:
    print(num, "->", additive_backtracking(num), "(kỳ vọng", expected, ")")

112358 -> True (kỳ vọng True )
199100199 -> True (kỳ vọng True )


In [10]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def isAdditiveNumber(self, num: str) -> bool:
        def dfs(a: int, b: int, num: str) -> bool:
            if not num:
                return True
            if a + b > 0 and num[0] == "0":
                return False
            for i in range(1, len(num) + 1):
                if a + b == int(num[:i]):
                    if dfs(b, a + b, num[i:]):
                        return True
                elif int(num[:i]) > a + b:
                    break
            return False

        n = len(num)
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                if i > 1 and num[0] == "0":
                    break
                if j - i > 1 and num[i] == "0":
                    continue
                if dfs(int(num[:i]), int(num[i:j]), num[j:]):
                    return True
        return False


print(Solution().isAdditiveNumber("112358"), Solution().isAdditiveNumber("199100199"))

True True


**Độ phức tạp Cách 3:** `O(n²)` cặp số đầu, mỗi cặp chuỗi đệ quy một nhánh (số tiếp theo luôn xác định) với `O(n)` bước, mỗi bước thử tối đa `O(n)` tiền tố (nhưng dừng sớm khi `value > target`): **Time `O(n³)`** (có thể tính `O(n⁴)` do đổi `int` cho tiền tố), **Space `O(n)`** (độ sâu đệ quy). Với `n <= 35` rất nhẹ.

### Cách 4 — Cắt tỉa theo độ dài: chỉ thử hai độ dài khả dĩ cho số kế tiếp

**Giải thích.** Số `c = a + b` có độ dài `max(len(a), len(b))` hoặc `max(len(a), len(b)) + 1` (khi có nhớ). Do đó thay vì thử mọi tiền tố, chỉ so `rest` với `str(a + b)` bằng `startswith` — chính là Cách 2; ta thêm **cắt tỉa sớm** các cặp `(i, j)` có `max(i, j) > n - i - j`.
#### Code demo

In [11]:
def additive_pruned(num):
    n = len(num)
    for i in range(1, n // 2 + 1):
        if num[0] == "0" and i > 1:
            break
        for j in range(1, n):
            if max(i, j) > n - i - j:                         # phần còn lại ngắn hơn số dài nhất trong hai số đầu
                break
            if num[i] == "0" and j > 1:
                break
            a, b = int(num[:i]), int(num[i:i + j])
            pos = i + j
            while pos < n:
                a, b = b, a + b                                   # số kế tiếp buộc phải là a + b
                text = str(b)
                if not num.startswith(text, pos):
                    break
                pos += len(text)
            if pos == n:
                return True
    return False


for num, expected in examples:
    print(num, "->", additive_pruned(num), "(kỳ vọng", expected, ")")

112358 -> True (kỳ vọng True )
199100199 -> True (kỳ vọng True )


**Độ phức tạp Cách 4:** vẫn **`O(n³)`** worst nhưng vòng `j` dừng sớm nhờ cắt tỉa ⇒ hằng số nhỏ hơn rõ rệt.

### Cách 5 — Cộng chuỗi, không đổi sang số nguyên (trả lời follow-up tràn số)

**Giải thích.** Giống Cách 2 nhưng giữ `a`, `b` là **chuỗi** và dùng `add_strings` (mục 3.5) để cộng. Không phụ thuộc kích thước số nguyên của ngôn ngữ (đúng cho C++/Java nơi `long` chỉ chứa ~`19` chữ số).
#### Code demo

In [12]:
def additive_string_add(num):
    n = len(num)
    for i in range(1, n):
        if num[0] == "0" and i > 1:
            break
        for j in range(1, n - i):
            if num[i] == "0" and j > 1:
                break
            if n - i - j < max(i, j):
                continue
            a, b = num[:i], num[i:i + j]
            pos = i + j
            while pos < n:
                c = add_strings(a, b)
                if not num.startswith(c, pos):
                    break
                pos += len(c)
                a, b = b, c
            if pos == n:
                return True
    return False


for num, expected in examples:
    print(num, "->", additive_string_add(num), "(kỳ vọng", expected, ")")

112358 -> True (kỳ vọng True )
199100199 -> True (kỳ vọng True )


**Độ phức tạp Cách 5:** **Time `O(n³)`** (mỗi `add_strings` `O(độ dài)`), **Space `O(n)`**; không dùng số nguyên lớn.

> Cách 2/3 là bản nộp phổ biến trong Python; Cách 5 là câu trả lời đúng tinh thần follow-up ở ngôn ngữ có số nguyên cố định.

## 8. TRACE từng bước

Ta trace Cách 2 trên `"199100199"` (in từng cặp độ dài thử, kết quả kiểm tra) và trên `"1023"` (false).
#### Code demo

In [13]:
def trace_two_first(num):
    n = len(num)
    print(f"num={num!r}, n={n}")
    for i in range(1, n):
        if num[0] == "0" and i > 1:
            print(f"  i={i}: số thứ nhất có số 0 đứng đầu -> dừng")
            break
        for j in range(1, n - i):
            if num[i] == "0" and j > 1:
                print(f"  (i={i}, j={j}): số thứ hai {num[i:i + j]!r} có số 0 đứng đầu -> bỏ")
                break
            if n - i - j < max(i, j):
                print(f"  (i={i}, j={j}): phần còn lại {n - i - j} ký tự < max({i},{j}) -> bỏ")
                continue
            a, b = int(num[:i]), int(num[i:i + j])
            pos, x, y = i + j, a, b
            steps = [f"{a}, {b}"]
            ok = True
            while pos < n:
                z = x + y
                if not num.startswith(str(z), pos):
                    steps.append(f"mong đợi {z} nhưng thấy {num[pos:pos + len(str(z))]!r}")
                    ok = False
                    break
                steps.append(str(z))
                pos += len(str(z))
                x, y = y, z
            print(f"  (i={i}, j={j}): dãy {', '.join(steps)} -> {'KHỚP HẾT' if ok and pos == n else 'sai'}")
            if ok and pos == n:
                return True
    return False


print(trace_two_first("199100199"))
print()
print(trace_two_first("1023"))

num='199100199', n=9
  (i=1, j=1): dãy 1, 9, mong đợi 10 nhưng thấy '91' -> sai
  (i=1, j=2): dãy 1, 99, 100, 199 -> KHỚP HẾT
True

num='1023', n=4
  (i=1, j=1): dãy 1, 0, mong đợi 1 nhưng thấy '2' -> sai
  (i=1, j=2): số thứ hai '02' có số 0 đứng đầu -> bỏ
  (i=2, j=1): phần còn lại 1 ký tự < max(2,1) -> bỏ
False


Bảng tóm tắt (`"199100199"`):

| `(i, j)` | `a`, `b` | số mong đợi | kết quả |
|---|---|---|---|
| `(1, 1)` | `1`, `9` | `10` vs `"91..."` | sai |
| `(1, 2)` | `1`, `99` | `100`, rồi `199` | **khớp hết ⇒ True** |

Với `"1023"` mọi cặp đều thất bại ⇒ `False`.

## 9. Code nộp LeetCode

Bản nộp dùng quay lui (Cách 3). Đúng `class Solution`, đúng `isAdditiveNumber(self, num: str) -> bool`.

In [14]:
class Solution:
    def isAdditiveNumber(self, num: str) -> bool:
        def dfs(a: int, b: int, num: str) -> bool:
            if not num:                                   # đã dùng hết chuỗi: dãy hợp lệ (đã có >= 3 số)
                return True
            if a + b > 0 and num[0] == "0":               # số tiếp theo > 0 không được bắt đầu bằng '0'
                return False
            for i in range(1, len(num) + 1):
                if a + b == int(num[:i]):                 # số tiếp theo buộc phải bằng a + b
                    if dfs(b, a + b, num[i:]):
                        return True
                elif int(num[:i]) > a + b:                # tiền tố đã lớn hơn mục tiêu: dừng
                    break
            return False

        n = len(num)
        for i in range(1, n - 1):                         # độ dài số thứ nhất
            for j in range(i + 1, n):                     # num[i:j] là số thứ hai
                if i > 1 and num[0] == "0":               # số thứ nhất có số 0 đứng đầu
                    break
                if j - i > 1 and num[i] == "0":           # số thứ hai có số 0 đứng đầu
                    continue
                if dfs(int(num[:i]), int(num[i:j]), num[j:]):
                    return True
        return False


sol = Solution()
for num, expected in examples:
    assert sol.isAdditiveNumber(num) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Hai số đầu xác định mọi thứ:** nếu dãy `x_1, x_2, x_3, ...` là cộng tính thì `x_3 = x_1 + x_2`, `x_4 = x_2 + x_3`, ... — được xác định duy nhất bởi `x_1, x_2`. Chuỗi là số cộng tính ⟺ tồn tại cách chọn độ dài `(i, j)` của hai số đầu sao cho dãy sinh ra bởi chúng, viết liền các chữ số, **trùng khít đúng toàn bộ chuỗi** (và có ít nhất ba số).
- **Vét mọi `(i, j)`:** vòng lặp xét mọi cặp độ dài hợp lệ; không bỏ sót vì dãy nào cũng bắt đầu bằng hai số có độ dài như vậy.
- **Số 0 đứng đầu:** kiểm tra trên **chuỗi** (không phải trên số đã đổi): số có độ dài `> 1` mà ký tự đầu là `'0'` bị loại; số `0` đơn lẻ vẫn hợp lệ, và tổng `a + b = 0` (hai số `0`) cho số tiếp theo `"0"` hợp lệ.
- **Đủ ba số:** với `i + j <= n - 1` luôn còn ít nhất một chữ số cho số thứ ba; khi `pos == n` mà không thất bại, ít nhất một số thứ ba đã khớp.
- **Cắt tỉa:** số thứ ba `a + b` có độ dài `>= max(len(a), len(b))` (không có số 0 đứng đầu và `a + b >= max(a, b)`), nên phần còn lại phải dài ít nhất `max(i, j)`.
- **Cộng chuỗi (Cách 5):** `add_strings` cộng đúng theo quy tắc cộng tay (nhớ `carry`), cho cùng kết quả như `str(int(x) + int(y))`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(num)` (tối đa `35`).

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi cách cắt | `O(2^n · n)` | `O(n)` |
| 2. Hai số đầu + kiểm tra lặp | `O(n³)` | `O(n)` |
| 3. Quay lui đệ quy | `O(n³)`–`O(n⁴)` | `O(n)` |
| 4. Cắt tỉa theo độ dài | `O(n³)` (hằng nhỏ) | `O(n)` |
| 5. Cộng chuỗi | `O(n³)` | `O(n)` |

Với `n = 35`: số cặp `(i, j)` khả dĩ cỡ `~ n²/4 ≈ 300`, mỗi cặp kiểm tra `≤ 35` bước ⇒ cỡ `10^4` thao tác — tức thời. Cách 1 là `~10^10` (không chạy được). Không có khác biệt average/worst về bậc.

## 12. Bộ test

`oracle` là brute force độc lập cho `n <= 13`: **liệt kê mọi cách cắt** (như Cách 1) nhưng kiểm tra bằng cách dựng dãy bằng **cộng chuỗi từng chữ số** `add_strings` (không dùng `int`) để độc lập với Cách 1. Với `n` lớn hơn, ta dựng chuỗi cộng tính **có chủ đích** (đáp án `True` chắc chắn) và đối chiếu các cách còn lại.
#### Code demo

In [15]:
def oracle(num):
    n = len(num)
    for cuts in range(1 << (n - 1)):
        parts, start = [], 0
        for k in range(n - 1):
            if cuts >> k & 1:
                parts.append(num[start:k + 1])
                start = k + 1
        parts.append(num[start:])
        if len(parts) < 3 or any(p[0] == "0" and len(p) > 1 for p in parts):
            continue
        if all(parts[k] == add_strings(parts[k - 2], parts[k - 1]) for k in range(2, len(parts))):
            return True
    return False


impls = {
    "all_splits": additive_all_splits,
    "two_first": additive_two_first,
    "backtracking": additive_backtracking,
    "pruned": additive_pruned,
    "string_add": additive_string_add,
    "submit": sol.isAdditiveNumber,
}

for num, expected in examples:
    assert oracle(num) == expected
    for key, fn in impls.items():
        assert fn(num) == expected, (key, num, fn(num), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [16]:
edge_cases = [
    ("1", False, "một chữ số: không đủ ba số"),
    ("12", False, "hai chữ số: chỉ hai số"),
    ("123", True, "1 + 2 = 3"),
    ("000", True, "0 + 0 = 0 (các số 0 đơn lẻ hợp lệ)"),
    ("0000", True, "0, 0, 0, 0"),
    ("101", True, "1, 0, 1"),
    ("1023", False, "1, 02, 3 có số 0 đứng đầu; 10, 2, 3 sai"),
    ("011235", True, "0, 1, 1, 2, 3, 5"),
    ("0235813", False, "bắt đầu 02: số thứ nhất có số 0 đứng đầu"),
    ("112358", True, "ví dụ 1"),
    ("199100199", True, "ví dụ 2"),
    ("1203", False, "12 + 0 != 3; 1 + 2 != 03; 1 + 20 != 3"),
    ("111", False, "1 + 1 = 2, không phải 1"),
    ("11235813", True, "Fibonacci tới 13"),
    ("12358132134", True, "1, 2, 3, 5, 8, 13, 21, 34"),
    ("1991000199", False, "1, 99, 100 rồi còn \"0199\" có số 0 đứng đầu"),
]
for num, expected, label in edge_cases:
    if len(num) <= 13:
        assert oracle(num) == expected, (label, oracle(num), expected)
    for key, fn in impls.items():
        if key == "all_splits" and len(num) > 13:
            continue
        assert fn(num) == expected, (key, label, fn(num), expected)
    print("PASS", label)

PASS một chữ số: không đủ ba số
PASS hai chữ số: chỉ hai số
PASS 1 + 2 = 3
PASS 0 + 0 = 0 (các số 0 đơn lẻ hợp lệ)
PASS 0, 0, 0, 0
PASS 1, 0, 1
PASS 1, 02, 3 có số 0 đứng đầu; 10, 2, 3 sai
PASS 0, 1, 1, 2, 3, 5
PASS bắt đầu 02: số thứ nhất có số 0 đứng đầu
PASS ví dụ 1
PASS ví dụ 2
PASS 12 + 0 != 3; 1 + 2 != 03; 1 + 20 != 3
PASS 1 + 1 = 2, không phải 1
PASS Fibonacci tới 13
PASS 1, 2, 3, 5, 8, 13, 21, 34
PASS 1, 99, 100 rồi còn "0199" có số 0 đứng đầu


### 12.2. Tính chất và phản ví dụ

- Chuỗi **sinh từ một dãy cộng tính** (hai số đầu bất kỳ không có số 0 đứng đầu, độ dài `>= 3` số) **luôn** cho `True`.
- Thêm/bớt **một chữ số ở cuối** (làm sai tổng cuối) thường chuyển thành `False` (so với oracle).
- **Phản ví dụ** cho lỗi cho phép số 0 đứng đầu: `"1023"` cho `True` nếu coi `"02"` hợp lệ (hiểu `1, 02, 3` với `int("02") = 2`).
- **Phản ví dụ** cho lỗi chấp nhận dãy hai số: `"12"` cho `True` nếu không kiểm tra `>= 3` số.
#### Code demo

In [17]:
def build_additive(first, second, length_limit=35):
    seq = [first, second]
    text = f"{first}{second}"
    while True:
        nxt = seq[-1] + seq[-2]
        if len(text) + len(str(nxt)) > length_limit:
            break
        seq.append(nxt)
        text += str(nxt)
    return text, seq


rng = random.Random(306)
generated = 0
for _ in range(500):
    a = rng.randint(1, 10 ** rng.randint(1, 8))
    b = rng.randint(1, 10 ** rng.randint(1, 8))
    text, seq = build_additive(a, b)
    if len(seq) >= 3:
        generated += 1
        assert sol.isAdditiveNumber(text), (text, seq)
print(f"PASS {generated} chuỗi sinh từ dãy cộng tính (độ dài tới 35, số rất lớn): luôn True")


def sai_cho_phep_so_0_dau(num):
    n = len(num)
    for i in range(1, n):
        for j in range(1, n - i):
            a, b = int(num[:i]), int(num[i:i + j])                    # SAI: không kiểm tra số 0 đứng đầu
            pos = i + j
            while pos < n:
                a, b = b, a + b
                if not num.startswith(str(b), pos):
                    break
                pos += len(str(b))
            if pos == n:
                return True
    return False


def sai_chap_nhan_hai_so(num):
    n = len(num)
    for i in range(1, n):
        if num[0] == "0" and i > 1:
            break
        for j in range(1, n - i + 1):                                 # SAI: cho phép j kéo tới hết chuỗi (chỉ hai số)
            if num[i] == "0" and j > 1:
                break
            a, b = int(num[:i]), int(num[i:i + j])
            pos = i + j
            while pos < n:
                a, b = b, a + b
                if not num.startswith(str(b), pos):
                    break
                pos += len(str(b))
            if pos == n:
                return True
    return False


print("'1023' đúng:", sol.isAdditiveNumber("1023"), "| cho phép 0 đứng đầu:", sai_cho_phep_so_0_dau("1023"))
print("'12'   đúng:", sol.isAdditiveNumber("12"), "| chấp nhận hai số:", sai_chap_nhan_hai_so("12"))
assert sai_cho_phep_so_0_dau("1023") is True and sai_chap_nhan_hai_so("12") is True

PASS 500 chuỗi sinh từ dãy cộng tính (độ dài tới 35, số rất lớn): luôn True
'1023' đúng: False | cho phép 0 đứng đầu: True
'12'   đúng: False | chấp nhận hai số: True


### 12.3. Đối chiếu ngẫu nhiên (có seed)
#### Code demo

In [18]:
rng = random.Random(4306)
total_true = 0
for trial in range(2500):
    n = rng.randint(1, 13)
    alphabet = rng.choice(["01", "012", "0123456789", "12358"])
    num = "".join(rng.choice(alphabet) for _ in range(n))
    truth = oracle(num)
    total_true += truth
    for key, fn in impls.items():
        assert fn(num) == truth, (key, num, fn(num), truth)
print(f"PASS 2500 chuỗi ngẫu nhiên (trong đó {total_true} là số cộng tính): mọi cách khớp oracle")

# Chuỗi dài (tới 35): dựng cộng tính (True) và biến đổi một chữ số (đối chiếu các cách nhanh với nhau)
checked = 0
for _ in range(400):
    a = rng.randint(1, 10 ** rng.randint(1, 12))
    b = rng.randint(1, 10 ** rng.randint(1, 12))
    text, seq = build_additive(a, b)
    if len(seq) < 3:
        continue
    variants = [text]
    pos = rng.randrange(len(text))
    variants.append(text[:pos] + str((int(text[pos]) + rng.randint(1, 9)) % 10) + text[pos + 1:])
    variants.append(text + str(rng.randint(0, 9)))
    for v in variants:
        if len(v) > 35:
            continue
        ref = additive_string_add(v)
        for key in ("two_first", "backtracking", "pruned", "submit"):
            assert impls[key](v) == ref, (key, v)
        checked += 1
print(f"PASS {checked} chuỗi dài (tới 35 chữ số): các cách nhanh khớp nhau (và khớp Cách 5 cộng chuỗi)")
t0 = time.perf_counter()
for _ in range(1000):
    sol.isAdditiveNumber("1" * 35)
print(f"1000 lần chuỗi xấu '1' * 35: {(time.perf_counter() - t0) * 1000:.1f} ms")

PASS 2500 chuỗi ngẫu nhiên (trong đó 69 là số cộng tính): mọi cách khớp oracle


PASS 1152 chuỗi dài (tới 35 chữ số): các cách nhanh khớp nhau (và khớp Cách 5 cộng chuỗi)


1000 lần chuỗi xấu '1' * 35: 1257.6 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cho phép số có số 0 đứng đầu
Xem `sai_cho_phep_so_0_dau` ở phần test: `"1023"` cho `True` (hiểu `1, 02, 3`). **Vì sao sai:** `int("02")` bằng `2` che mất số 0 đứng đầu; đề cấm. **Code đúng:** kiểm tra trên **chuỗi**: `len(t) > 1 and t[0] == "0"` ⇒ loại.

### Lỗi 2: Chấp nhận dãy chỉ có hai số
Xem `sai_chap_nhan_hai_so`: `"12"` cho `True` vì `pos == n` ngay sau hai số. **Vì sao sai:** đề cần **ít nhất ba số**. **Code đúng:** giới hạn `i + j <= n - 1` (còn chỗ cho số thứ ba).

### Lỗi 3: So sánh tiền tố mà không kiểm tra hết chuỗi
#### Code sai

In [19]:
def sai_khong_het_chuoi(num):
    n = len(num)
    for i in range(1, n):
        for j in range(1, n - i):
            if (num[0] == "0" and i > 1) or (num[i] == "0" and j > 1):
                continue
            a, b = int(num[:i]), int(num[i:i + j])
            c = a + b
            if num.startswith(str(c), i + j):                    # SAI: chỉ kiểm tra số thứ ba, quên phần còn lại
                return True
    return False


for num in ("1234", "1123"):
    print(num, "đúng:", sol.isAdditiveNumber(num), "| sai:", sai_khong_het_chuoi(num))

1234 đúng: False | sai: True
1123 đúng: True | sai: True


**Vì sao sai:** `"1234"`: `1 + 2 = 3`, tiền tố `"3"` khớp nhưng còn dư `"4"` (và `2 + 3 = 5 ≠ 4`). Phải tiếp tục kiểm tra tới khi **dùng hết** chuỗi.

### Lỗi 4: Cận độ dài sai (bỏ sót hoặc quá chặt)
#### Code sai

In [20]:
def sai_can_do_dai(num):
    n = len(num)
    for i in range(1, n // 3 + 1):                               # SAI: quá chặt, số thứ nhất có thể dài hơn n/3
        for j in range(1, n // 3 + 1):
            if (num[0] == "0" and i > 1) or (num[i] == "0" and j > 1):
                continue
            a, b = int(num[:i]), int(num[i:i + j])
            pos = i + j
            while pos < n:
                a, b = b, a + b
                if not num.startswith(str(b), pos):
                    break
                pos += len(str(b))
            if pos == n:
                return True
    return False


for num in ("99911000", "112358"):
    print(num, "đúng:", sol.isAdditiveNumber(num), "| sai:", sai_can_do_dai(num))

99911000 đúng: True | sai: False
112358 đúng: True | sai: True


**Vì sao sai:** `"99911000"` là dãy `999, 1, 1000` (`999 + 1 = 1000`): số thứ nhất có `3` chữ số nhưng `n // 3 = 2`, nên cận `i <= n // 3` loại nhầm nó. Cận đúng là `max(i, j) <= n - i - j` (phần còn lại đủ dài cho số thứ ba), không phải chia ba chiều dài.

### Lỗi 5: Dùng `int` ở ngôn ngữ có số nguyên cố định (tràn số)
Với chuỗi tới `35` chữ số, số có thể vượt `long`; trong C++/Java phải cộng chuỗi (Cách 5). Python không tràn nhưng hiểu lý do rất quan trọng khi phỏng vấn.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**cắt chuỗi thành các phần thỏa quan hệ giữa các phần liên tiếp**" mà quan hệ xác định phần sau từ các phần trước → **chọn các phần đầu rồi kiểm tra** (số lựa chọn đầu nhỏ).
- Quan hệ kiểu Fibonacci `x_k = x_{k-1} + x_{k-2}`: **hai số đầu xác định tất cả**.
- **Số 0 đứng đầu**: kiểm tra trên **chuỗi** (`len > 1 and s[0] == '0'`), không trên số nguyên.
- Luôn nhớ điều kiện **tối thiểu số phần tử** (`>= 3`).
- Số lớn → **cộng chuỗi** hoặc số nguyên lớn của ngôn ngữ.
- Họ hàng: **Split Array into Fibonacci Sequence** (842), **Restore IP Addresses** (93), **Palindrome Partitioning** (131), **Decode Ways** (91).
- Thử: `"000"`, `"101"`, `"1023"`, `"12"`, chuỗi Fibonacci dài.

## 15. Biến thể của bài

1. **Split Array into Fibonacci Sequence** (LeetCode 842): trả về **chính dãy** (có giới hạn `2^31 - 1`).
2. **Số cộng tính có thể thay "cộng" bằng "nhân" / hiệu**: đổi phép toán.
3. **Đếm** số cách cắt thành dãy cộng tính (không chỉ có/không).
4. **Chuỗi dài hơn** (`10^3`): cần cắt tỉa theo độ dài và so sánh chuỗi rất cẩn thận.
5. **Trả về dãy đầu tiên** tìm được.
#### Code demo

In [21]:
def additive_sequence(num):
    """Trả về một dãy cộng tính (danh sách số) hoặc []."""
    n = len(num)
    for i in range(1, n):
        if num[0] == "0" and i > 1:
            break
        for j in range(1, n - i):
            if num[i] == "0" and j > 1:
                break
            if n - i - j < max(i, j):
                continue
            seq = [int(num[:i]), int(num[i:i + j])]
            pos = i + j
            while pos < n:
                nxt = seq[-1] + seq[-2]
                if not num.startswith(str(nxt), pos):
                    break
                seq.append(nxt)
                pos += len(str(nxt))
            if pos == n:
                return seq
    return []


def count_additive_splits(num):
    """Đếm số dãy cộng tính (khác nhau theo cặp độ dài số đầu) bằng cách xét mọi (i, j)."""
    n = len(num)
    count = 0
    for i in range(1, n):
        for j in range(1, n - i):
            if (num[0] == "0" and i > 1) or (num[i] == "0" and j > 1):
                continue
            a, b, pos = int(num[:i]), int(num[i:i + j]), i + j
            while pos < n:
                a, b = b, a + b
                if not num.startswith(str(b), pos):
                    break
                pos += len(str(b))
            if pos == n:
                count += 1
    return count


print(additive_sequence("199100199"), additive_sequence("1023"), count_additive_splits("112358"), count_additive_splits("101"))
# [1, 99, 100, 199] [] 1 1

[1, 99, 100, 199] [] 1 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cách cắt | `2^(n-1)` cách, kiểm tra từng cách | `O(2^n·n)` | `O(n)` | oracle, `n <= 16` |
| 2. Hai số đầu + lặp | `(i, j)` rồi dựng dãy, `startswith` | `O(n³)` | `O(n)` | gọn, dễ hiểu |
| 3. Quay lui đệ quy | `dfs(a, b, rest)` | `O(n³)` | `O(n)` | **bản nộp chuẩn** |
| 4. Cắt tỉa theo độ dài | bỏ cặp có `max(i, j) > phần còn lại` | `O(n³)` hằng nhỏ | `O(n)` | tăng tốc |
| 5. Cộng chuỗi | `add_strings` thay `int` | `O(n³)` | `O(n)` | follow-up tràn số |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `valid_number("0")` và `valid_number("07")`; (b) `add_strings("999", "1")`; (c) `Solution().isAdditiveNumber("12")`; (d) `Solution().isAdditiveNumber("101")`.

**Bài 2 (tính bằng tay).** Chứng minh bằng tay `"11235813"` là số cộng tính (liệt kê dãy) và `"1235813"` thì sao (thử các cặp độ dài `(1,1)`, `(1,2)`, `(2,1)`).

**Bài 3 (điền TODO).** Hoàn thiện `additive_todo(num)` theo Cách 2.
#### Code demo

In [22]:
def additive_todo(num):
    n = len(num)
    # TODO 1: for i in range(1, n): (bỏ nếu num[0] == "0" và i > 1)
    # TODO 2: for j in range(1, n - i): (bỏ nếu num[i] == "0" và j > 1; bỏ nếu n - i - j < max(i, j))
    # TODO 3: a, b = int(num[:i]), int(num[i:i + j]); pos = i + j
    # TODO 4: while pos < n: z = a + b; nếu not num.startswith(str(z), pos) thì break; pos += len(str(z)); a, b = b, z
    # TODO 5: nếu pos == n trả True; cuối cùng trả False
    pass


# Tự làm xong rồi mới mở test:
# for num, expected in examples:
#     assert additive_todo(num) == expected

**Bài 4 (tự viết lại).** Viết `split_fibonacci(num)` (LeetCode 842): trả một dãy Fibonacci (mỗi số trong `[0, 2^31 - 1]`) hoặc `[]`; bạn có thể dùng lại ý tưởng Cách 2/3 và thêm giới hạn kích thước.

**Bài 5 (biến thể / mini challenge).** Viết `additive_number_general(num, op)` cho phép đổi phép toán: `op = lambda x, y: x + y` (Fibonacci), hoặc `lambda x, y: x * y`, `abs(x - y)`. Chú ý điều kiện dừng khi giá trị không tăng và số `0`; kiểm tra rằng với `op = add` kết quả giống `Solution().isAdditiveNumber`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: số `0` đơn lẻ hợp lệ, `"07"` thì không; cộng chuỗi `999 + 1 = 1000`. Bài 2: thử `(1,1)`: `1 + 1 = 2`, ...
- **Mức 2 (vừa):** Bài 1: `True`, `False`, `"1000"`; `False`, `True` (1, 0, 1). Bài 2: `"11235813"`: `1, 1, 2, 3, 5, 8, 13`; `"1235813"`: `1, 2, 3, 5, 8, 13` ⇒ cũng là dãy cộng tính (`1+2=3`, `2+3=5`, `3+5=8`, `5+8=13`). Bài 3: khung Cách 2.
- **Mức 3 (sâu):** Bài 4: thêm điều kiện mỗi số `<= 2^31 - 1` (và giữ quy tắc số 0 đứng đầu); trả danh sách số. Bài 5: thay phép `+` bằng `op(x, y)` trong vòng lặp sinh số tiếp theo; điều kiện cắt tỉa theo độ dài không còn đúng với mọi phép toán — thử không cắt tỉa.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [23]:
# Bài 1
assert valid_number("0") and not valid_number("07")
assert add_strings("999", "1") == "1000"
assert Solution().isAdditiveNumber("12") is False and Solution().isAdditiveNumber("101") is True

# Bài 2
assert additive_sequence("11235813") == [1, 1, 2, 3, 5, 8, 13]
assert additive_sequence("1235813") == [1, 2, 3, 5, 8, 13]


# Bài 3
def additive_todo_ans(num):
    n = len(num)
    for i in range(1, n):
        if num[0] == "0" and i > 1:
            break
        for j in range(1, n - i):
            if num[i] == "0" and j > 1:
                break
            if n - i - j < max(i, j):
                continue
            a, b = int(num[:i]), int(num[i:i + j])
            pos = i + j
            while pos < n:
                z = a + b
                if not num.startswith(str(z), pos):
                    break
                pos += len(str(z))
                a, b = b, z
            if pos == n:
                return True
    return False


for num, expected in examples:
    assert additive_todo_ans(num) == expected


# Bài 4
def split_fibonacci(num):
    limit = 2 ** 31 - 1
    n = len(num)
    for i in range(1, n):
        if num[0] == "0" and i > 1:
            break
        for j in range(1, n - i):
            if num[i] == "0" and j > 1:
                break
            a, b = int(num[:i]), int(num[i:i + j])
            if a > limit or b > limit:
                continue
            seq, pos = [a, b], i + j
            while pos < n:
                z = seq[-1] + seq[-2]
                if z > limit or not num.startswith(str(z), pos):
                    break
                seq.append(z)
                pos += len(str(z))
            if pos == n and len(seq) >= 3:
                return seq
    return []


assert split_fibonacci("123456579") == [123, 456, 579]
assert split_fibonacci("11235813") == [1, 1, 2, 3, 5, 8, 13]
assert split_fibonacci("112358130") == []


# Bài 5
def additive_number_general(num, op):
    n = len(num)
    for i in range(1, n):
        if num[0] == "0" and i > 1:
            break
        for j in range(1, n - i):
            if num[i] == "0" and j > 1:
                break
            a, b = int(num[:i]), int(num[i:i + j])
            pos = i + j
            while pos < n:
                z = op(a, b)
                if not num.startswith(str(z), pos):
                    break
                pos += len(str(z))
                a, b = b, z
            if pos == n and i + j < n:
                return True
    return False


rng = random.Random(22)
for _ in range(500):
    s = "".join(rng.choice("0123456789") for _ in range(rng.randint(1, 12)))
    assert additive_number_general(s, lambda x, y: x + y) == sol.isAdditiveNumber(s), s
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai số đầu + quay lui kiểm tra)

```python
class Solution:
    def isAdditiveNumber(self, num: str) -> bool:
        def dfs(a, b, rest):
            if not rest:
                return True
            if a + b > 0 and rest[0] == "0":
                return False
            for i in range(1, len(rest) + 1):
                if a + b == int(rest[:i]):
                    if dfs(b, a + b, rest[i:]):
                        return True
                elif int(rest[:i]) > a + b:
                    break
            return False

        n = len(num)
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                if i > 1 and num[0] == "0":
                    break
                if j - i > 1 and num[i] == "0":
                    continue
                if dfs(int(num[:i]), int(num[i:j]), num[j:]):
                    return True
        return False
```

**Checklist**

- [ ] Chọn **hai số đầu** (độ dài `i`, `j`) rồi **kiểm tra** (phần còn lại bị xác định)?
- [ ] Còn **ít nhất một chữ số** cho số thứ ba (dãy có `>= 3` số)?
- [ ] Số có **số 0 đứng đầu** kiểm tra trên **chuỗi** (`"0"` đơn lẻ hợp lệ, `"03"` không)?
- [ ] Phải **dùng hết** chuỗi (`pos == n` / `rest` rỗng), không chỉ khớp tiền tố?
- [ ] Biết follow-up: số lớn ⇒ **cộng chuỗi**?
- [ ] Đã thử `"000"`, `"101"`, `"1023"`, `"12"`, Fibonacci dài, số 35 chữ số?
- [ ] Đã đối chiếu với brute force mọi cách cắt trên chuỗi ngắn ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def isAdditiveNumber(self, num: str) -> bool`?